# Disaster Detection - 10-Minute Fast Training Workflow\n
\n
This notebook provides a ready-to-use workflow to train the disaster detection model quickly with GPU and then move the trained model back to the Antigravity AI backend.\n
\n
Using a lightweight model like **MobileNetV2**, training usually finishes in **~8–10 minutes on GPU**.

## 1️⃣ Enable GPU\n
1. Open **Google Colab**\n
2. Click `Runtime → Change runtime type`\n
3. Select `Hardware accelerator → GPU`

In [2]:
import tensorflow as tf\n
print("GPU Available:", tf.config.list_physical_devices('GPU'))

SyntaxError: unexpected character after line continuation character (1265304562.py, line 1)

## 2️⃣ Mount Dataset from Google Drive\n
Make sure your dataset structure looks like this:\n
```\n
dataset/\n
   fire/\n
   flood/\n
   landslide/\n
   earthquake/\n
   normal/\n
```

In [4]:
from google.colab import drive\n
drive.mount('/content/drive')

SyntaxError: unexpected character after line continuation character (1942167535.py, line 1)

## 3️⃣ Install Required Libraries

In [5]:
!pip install tensorflow matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 4️⃣ Load Dataset Automatically

In [ ]:
import tensorflow as tf\n
IMG_SIZE = (224, 224)\n
BATCH_SIZE = 32\n
\n
# Update this path to where your dataset is in Google Drive\n
DATASET_PATH = "/content/drive/MyDrive/disaster_dataset"\n
\n
train_ds = tf.keras.preprocessing.image_dataset_from_directory(\n
    DATASET_PATH,\n
    validation_split=0.2,\n
    subset="training",\n
    seed=123,\n
    image_size=IMG_SIZE,\n
    batch_size=BATCH_SIZE\n
)\n
\n
val_ds = tf.keras.preprocessing.image_dataset_from_directory(\n
    DATASET_PATH,\n
    validation_split=0.2,\n
    subset="validation",\n
    seed=123,\n
    image_size=IMG_SIZE,\n
    batch_size=BATCH_SIZE\n
)

## 5️⃣ Add Data Augmentation (Improves Accuracy)

In [ ]:
from tensorflow.keras import layers\n
\n
data_augmentation = tf.keras.Sequential([\n
    layers.RandomFlip("horizontal"),\n
    layers.RandomRotation(0.1),\n
    layers.RandomZoom(0.1)\n
])

## 6️⃣ Load Pretrained Lightweight Model

In [ ]:
from tensorflow.keras.applications import MobileNetV2\n
from tensorflow.keras import models\n
\n
base_model = MobileNetV2(\n
    input_shape=(224, 224, 3),\n
    include_top=False,\n
    weights="imagenet"\n
)\n
\n
# Freeze features to ensure very fast training\n
base_model.trainable = False

## 7️⃣ Add Classification Layers & ⃣ Compile

In [ ]:
x = base_model.output\n
x = tf.keras.layers.GlobalAveragePooling2D()(x)\n
x = tf.keras.layers.Dense(128, activation="relu")(x)\n
outputs = tf.keras.layers.Dense(5, activation="softmax")(x)\n
\n
model = models.Model(base_model.input, outputs)\n
\n
model.compile(\n
    optimizer="adam",\n
    loss="sparse_categorical_crossentropy",\n
    metrics=["accuracy"]\n
)

## 9️⃣ Train Model (GPU ~8 minutes)

In [ ]:
history = model.fit(\n
    train_ds,\n
    validation_data=val_ds,\n
    epochs=5\n
)

## 🔟 Save & Download Trained Model

In [ ]:
model.save("disaster_model.h5")\n
\n
from google.colab import files\n
files.download("disaster_model.h5")

## 1️⃣2️⃣ Convert to Lightweight TensorFlow Lite (Optional)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)\n
tflite_model = converter.convert()\n
\n
open("disaster_model.tflite", "wb").write(tflite_model)\n
files.download("disaster_model.tflite")